# DataGen Class Testing

This notebook is a small end-to-end smoke test for `DataGen`. It uses the simple ECM requested as `R-[P,R]-[P,R]`, labelled explicitly as `R1-[P2,R3]-[P4,R5]` so AutoEIS can map parameters unambiguously.

Run this notebook in the AutoREC environment. The setup cell searches from the current kernel working directory upward to find the `generate_data_pipline/` directory, so the notebook does not require a hard-coded launch directory.


## Example Command

Other users can run the same workflow from a shell with:

```bash
python - <<'PY'
from pathlib import Path
import shutil
import sys

def find_pipeline_dir(start=Path.cwd()):
    start = Path(start).resolve()
    for candidate in (start, *start.parents):
        nested = candidate / "generate_data_pipline"
        if (nested / "data_gen.py").exists():
            return nested
        if (candidate / "data_gen.py").exists():
            return candidate
    raise FileNotFoundError("Could not find generate_data_pipline.")

pipeline_dir = find_pipeline_dir()
repo_dir = pipeline_dir.parent
for path in (repo_dir / "src", repo_dir):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from generate_data_pipline.data_gen import DataGen

generator = DataGen(
    random_ecm_circuit="R1-[P2,R3]-[P4,R5]",
    output_dir=pipeline_dir / "data",
    n_random_candidates=500,
    max_selected_curves=20,
    excluded_simplified_ecms=("R1", "R1-C2"),
    verbose=True,
)

target_counts = {
    "R1-[P1,R2]": 55,
    "R1-[P1,R2]-P2": 100,
    "R1-[P1,R2]-[P2,R3]": 100,
}
balanced_df, _ = generator.generate_data(
    target_per_relabel=target_counts,
    target_relabel_ecms=tuple(target_counts),
    export=False,
)
final_df = generator.build_final_relabel_df(
    balanced_df,
    target_per_relabel=target_counts,
    target_relabel_ecms=tuple(target_counts),
)
data_dir = pipeline_dir / "data"
if data_dir.exists():
    shutil.rmtree(data_dir)
generator.export_eis_data_folder(final_df, output_dir=data_dir)
PY
```


In [34]:
from pathlib import Path
import importlib
import shutil
import sys

def find_pipeline_dir(start=Path.cwd()):
    """Find the generate_data_pipline directory containing data_gen.py."""
    start = Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "data_gen.py").exists() and (candidate / "ecm_simplification_functions").is_dir():
            return candidate
        nested = candidate / "generate_data_pipline"
        if (nested / "data_gen.py").exists() and (nested / "ecm_simplification_functions").is_dir():
            return nested
    raise FileNotFoundError(
        "Could not find generate_data_pipline from the current working "
        "directory or its parents. If your kernel starts elsewhere, set "
        "pipeline_dir manually to the directory containing data_gen.py."
    )

pipeline_dir = find_pipeline_dir()
repo_dir = pipeline_dir.parent
for path in (repo_dir / "src", repo_dir):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

try:
    import autoeis  # noqa: F401
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from tqdm.auto import tqdm  # noqa: F401
except ImportError as exc:
    raise ImportError("Run this notebook in the AutoREC environment.") from exc

import generate_data_pipline.data_gen as data_gen_module

data_gen_module = importlib.reload(data_gen_module)
DataGen = data_gen_module.DataGen

%matplotlib inline


In [35]:
source_ecm = "R1-[P2,R3]-[P4,R5]"
output_dir = pipeline_dir / "data"

generator = DataGen(
    random_ecm_circuit=source_ecm,
    output_dir=output_dir,
    n_random_candidates=500,
    max_selected_curves=20,
    excluded_simplified_ecms=("R1", "R1-C2"),
    fim_refit_max_iters=3,
    fim_refit_min_iters=1,
    fim_refit_max_nfev=100,
    verbose=True,
)

source_ecm


'R1-[P2,R3]-[P4,R5]'

## Generate a Small Dataset

Generate a small smoke-test dataset and build `final_df`. This cell does not write files.


In [36]:
balanced_df, batch_infos = generator.generate_data(
    target_per_relabel=1,
    min_batches=1,
    max_batches=5,
    seed_start=2026,
    n_random_candidates=500,
    max_selected_curves=20,
    export=False,
)

final_df = generator.build_final_relabel_df(
    balanced_df,
    target_per_relabel=1,
)

print(f"Generated rows: {len(final_df)}")
display(final_df[["global_position", "original_ecm", "simplified_ecm", "fim_relabel_ecm", "relabel_ecm"]])


Simulating random candidates:   0%|          | 0/500 [00:00<?, ?it/s]

Parser full simplify selected curves:   0%|          | 0/20 [00:00<?, ?it/s]

Dropping 0 excluded simplified ECMs
Keeping 20 rows after simplified ECM exclusion
Dropping 0 final relabelled ECMs with P-P series
Keeping 20 final relabelled ECMs
Dropping 0 invalid relabelled ECMs
Keeping 20 valid relabelled ECMs
Batch 1: first-batch target groups 3, discovered groups 3
Added this batch:
R1-[P1,R2]            1
R1-[P1,R2]-P2         1
R1-[P1,R2]-[P2,R3]    1
Accumulated target counts:
R1-[P1,R2]            1
R1-[P1,R2]-P2         1
R1-[P1,R2]-[P2,R3]    1
Generated relabel counts before final export trimming:
relabel_ecm
R1-[P1,R2]            1
R1-[P1,R2]-P2         1
R1-[P1,R2]-[P2,R3]    1
Generated rows: 3
Dropping 0 excluded simplified ECMs
Keeping 3 rows after simplified ECM exclusion
Generated rows: 3


,global_position,original_ecm,simplified_ecm,fim_relabel_ecm,relabel_ecm
0,0,"R1-[P2,R3]-[P4,R5]","R1-[C2,R3]","R1-[C2,R3]","R1-[P1,R2]"
1,1,"R1-[P2,R3]-[P4,R5]","R1-[P2,R3]-C4","R1-[P2,R3]-C4","R1-[P1,R2]-P2"
2,2,"R1-[P2,R3]-[P4,R5]","R1-[P2,R3]-[P4,R5]","R1-[P2,R3]-[P4,R5]","R1-[P1,R2]-[P2,R3]"


## Export Small Dataset

Clear `generate_data_pipline/data`, then write each EIS curve as both CSV and PNG under `data/<ECM>/csv` and `data/<ECM>/png`.


In [ ]:
if output_dir.exists():
    shutil.rmtree(output_dir)
output_dir.mkdir(parents=True, exist_ok=True)

data_dir = generator.export_eis_data_folder(final_df, output_dir=output_dir)
csv_files = sorted(data_dir.glob("*/csv/eis_*.csv"))
png_files = sorted(data_dir.glob("*/png/eis_*.png"))

print(f"Generated data folder saved to: {data_dir}")
print(f"CSV files written: {len(csv_files)}")
print(f"PNG files written: {len(png_files)}")
print(f"Example EIS CSV: {csv_files[0]}")
print(f"Example EIS PNG: {png_files[0]}")
display(pd.read_csv(csv_files[0]).head())


## Batch Metadata


In [37]:
pd.DataFrame(batch_infos)


,batch_id,seed,valid_candidate_count,filtered_candidate_count,selected_curve_count,reference_curve_count,after_final_filters_count,removed_by_high_frequency_filter,removed_by_excluded_simplified_ecm_filter
0,0,2026,500,317,20,0,20,183,0


## Generate Larger Batch

Generate the requested 55/100/100 dataset and build `batch_final_df`. This cell does not write files.


In [38]:
batch_generator = DataGen(
    random_ecm_circuit=source_ecm,
    output_dir=pipeline_dir / "data",
    n_random_candidates=500,
    max_selected_curves=20,
    excluded_simplified_ecms=("R1", "R1-C2"),
    fim_refit_max_iters=10,
    fim_refit_min_iters=1,
    fim_refit_max_nfev=100,
    verbose=True,
)

batch_target_counts = {
    "R1-[P1,R2]": 55,
    "R1-[P1,R2]-P2": 100,
    "R1-[P1,R2]-[P2,R3]": 100,
}

batch_balanced_df, batch_infos = batch_generator.generate_data(
    target_per_relabel=batch_target_counts,
    target_relabel_ecms=tuple(batch_target_counts),
    min_batches=1,
    max_batches=100,
    seed_start=2026,
    n_random_candidates=5000,
    max_selected_curves=150,
    export=False,
)

batch_final_df = batch_generator.build_final_relabel_df(
    batch_balanced_df,
    target_per_relabel=batch_target_counts,
    target_relabel_ecms=tuple(batch_target_counts),
)

print(f"Generated rows: {len(batch_final_df)}")
display(batch_final_df["relabel_ecm"].value_counts().sort_index())
display(pd.DataFrame(batch_infos))


Simulating random candidates:   0%|          | 0/5000 [00:00<?, ?it/s]

Parser full simplify selected curves:   0%|          | 0/150 [00:00<?, ?it/s]

Dropping 0 excluded simplified ECMs
Keeping 150 rows after simplified ECM exclusion
Dropping 0 final relabelled ECMs with P-P series
Keeping 150 final relabelled ECMs
Dropping 1 invalid relabelled ECMs
Keeping 149 valid relabelled ECMs
Batch 1: manual target groups 3, discovered groups 3
Added this batch:
R1-[P1,R2]             17
R1-[P1,R2]-P2          22
R1-[P1,R2]-[P2,R3]    100
Accumulated target counts:
R1-[P1,R2]             17
R1-[P1,R2]-P2          22
R1-[P1,R2]-[P2,R3]    100


Simulating random candidates:   0%|          | 0/5000 [00:00<?, ?it/s]

Parser full simplify selected curves:   0%|          | 0/150 [00:00<?, ?it/s]

...

Simulating random candidates:   0%|          | 0/5000 [00:00<?, ?it/s]

Parser full simplify selected curves:   0%|          | 0/150 [00:00<?, ?it/s]

Dropping 0 excluded simplified ECMs
Keeping 150 rows after simplified ECM exclusion
Dropping 0 final relabelled ECMs with P-P series
Keeping 150 final relabelled ECMs
Dropping 0 invalid relabelled ECMs
Keeping 150 valid relabelled ECMs
Batch 20: manual target groups 3, discovered groups 3
Added this batch:
R1-[P1,R2]            1
R1-[P1,R2]-P2         0
R1-[P1,R2]-[P2,R3]    0
Accumulated target counts:
R1-[P1,R2]             55
R1-[P1,R2]-P2         100
R1-[P1,R2]-[P2,R3]    100
Generated relabel counts before final export trimming:
relabel_ecm
R1-[P1,R2]             55
R1-[P1,R2]-P2         100
R1-[P1,R2]-[P2,R3]    100
Generated rows: 255
Dropping 0 excluded simplified ECMs
Keeping 255 rows after simplified ECM exclusion
Generated rows: 255


relabel_ecm
R1-[P1,R2]             55
R1-[P1,R2]-P2         100
R1-[P1,R2]-[P2,R3]    100
Name: count, dtype: int64

,batch_id,seed,valid_candidate_count,filtered_candidate_count,selected_curve_count,reference_curve_count,after_final_filters_count,removed_by_high_frequency_filter,removed_by_excluded_simplified_ecm_filter
0,0,2026,5000,3015,150,0,149,1985,0
1,1,2027,5000,2994,150,139,149,2006,0
2,2,2028,5000,3008,150,166,150,1992,0
3,3,2029,5000,3009,150,189,149,1991,0
4,4,2030,5000,3012,150,206,150,1988,0
5,5,2031,5000,2993,150,224,149,2007,0
6,6,2032,5000,3031,150,238,150,1969,0
7,7,2033,5000,3023,150,241,150,1977,0
8,8,2034,5000,2992,150,244,150,2008,0
9,9,2035,5000,3073,150,245,150,1927,0


## Export Larger Batch

Clear `generate_data_pipline/data`, then write `data/<ECM>/csv/eis_N.csv` and `data/<ECM>/png/eis_N.png` from `batch_final_df`.


In [39]:
batch_output_dir = pipeline_dir / "data"
if batch_output_dir.exists():
    shutil.rmtree(batch_output_dir)
batch_output_dir.mkdir(parents=True, exist_ok=True)

batch_data_dir = batch_generator.export_eis_data_folder(
    batch_final_df,
    output_dir=batch_output_dir,
)

batch_csv_files = sorted(batch_data_dir.glob("*/csv/eis_*.csv"))
batch_png_files = sorted(batch_data_dir.glob("*/png/eis_*.png"))
print(f"Generated data folder saved to: {batch_data_dir}")
print(f"CSV files written: {len(batch_csv_files)}")
print(f"PNG files written: {len(batch_png_files)}")
display(pd.read_csv(batch_csv_files[0]).head())


Generated data folder saved to: /Users/yinzhecheng/Documents/UT/MSE/AutoREC_fork/AutoREC/generate_data_pipline/data
CSV files written: 255
PNG files written: 255


,freq,Z_real,Z_imag
0,0.010000,27480.944682,-257.885299
1,0.012263,27451.872240,-283.072400
2,0.015039,27419.909982,-310.654458
3,0.018443,27384.772886,-340.845643
4,0.022617,27346.148372,-373.876421


# UMAP analysis

In [9]:
# from autorec.data_preparation import EISDataPrep
# import umap
# import seaborn as sns

In [10]:
# data_prep_training = EISDataPrep("../data/examples/training_dataset.pkl", mode="load")
# data_training = data_prep_training.load()

In [11]:
# data_prep_new = EISDataPrep(
#     path=pipeline_dir / "data" / "data",
#     mode="process",
#     evaluation=True,
# )
# data_new = data_prep_new.load()
# data_new.to_pickle(pipeline_dir / "data" / "data.pkl")
# print(data_prep_new.get_summary())

In [12]:
# flatten_Z_training = np.vstack(data_training.flatten_Z.values)
# flatten_Z_new = np.vstack(data_new.flatten_Z.values)
# reducer = umap.UMAP(random_state=42, transform_seed=42)
# reducer.fit(flatten_Z_training)

In [13]:
# embedding_training = reducer.transform(flatten_Z_training)
# embedding_new = reducer.transform(flatten_Z_new)

In [14]:
# plot_df_training = pd.DataFrame({
#     "UMAP 1": embedding_training[:, 0],
#     "UMAP 2": embedding_training[:, 1],
#     "ecm": data_training.true_circuit
# })
# plot_df_new = pd.DataFrame({
#     "UMAP 1": embedding_new[:, 0],
#     "UMAP 2": embedding_new[:, 1],
#     "ecm": data_new.true_circuit
# })

In [15]:
# plt.figure()
# sns.scatterplot(
#     data=plot_df_training, x="UMAP 1", y="UMAP 2", hue="ecm", s=25, alpha=0.8
# )
# sns.scatterplot(
#     data=plot_df_new,
#     x="UMAP 1",
#     y="UMAP 2",
#     hue="ecm",
#     marker="D",
#     edgecolor="black",
#     linewidth=1,
#     s=25,
#     alpha=0.8,
# )
# plt.legend(title="ECM", bbox_to_anchor=(1, 1))
# # plt.tight_layout()
# plt.show()